**1. LoRA (Low-Rank Adaptation)**

In [12]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM

In [18]:
# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,     # Rank (low-rank dimension)
    lora_alpha=32,  # Scaling factor
    lora_dropout=0.1,
    target_modules=["c_attn"],
    fan_in_fan_out=True
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained("gpt2")
model = get_peft_model(model, lora_config)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


In [22]:
model.print_trainable_parameters()
print(f"Total parameters: {model.num_parameters():}")

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364
Total parameters: 124734720


In [ ]:
trainer = Trainer(model=model,)
trainer.train()

model.save_pretrained("./lora_model") 

**2. Soft Prompts (Prompt Tuning)**

In [25]:
from peft import PromptTuningConfig, PromptTuningInit

In [26]:
prompt_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    num_virtual_tokens=20,  # Number of learnable prompt tokens
    prompt_tuning_init=PromptTuningInit.TEXT,  # Initialize from text
    prompt_tuning_init_text="Classify the sentiment of this text:",
    tokenizer_name_or_path="gpt2"
)

In [27]:
model = AutoModelForCausalLM.from_pretrained("gpt2")
model = get_peft_model(model, prompt_config)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [28]:
model.print_trainable_parameters()

trainable params: 15,360 || all params: 124,455,168 || trainable%: 0.0123


In [ ]:
trainer = Trainer(model=model,)
trainer.train()